# Part 3 · Notebook 03 — Generators, decorators and context managers

**Sessions:** S5 (Iterators, generators, decorators & context managers) · [Lesson plan](../../docs/lessons/PART_03_PYTHON_ENGINEERING.md) · graded labs in [`labs/part03/`](../../labs/part03/)

**You will:**
1. Stream ticks with a generator and turn them into bars without holding the day in memory.
2. Write a decorator that counts calls, keeping the function's name.
3. Time a block of code with your own context manager, and retry a flaky call.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p3lib.py is in notebooks/part03/
    sys.path.insert(0, str(d))
from decimal import Decimal
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p3lib as p

p.use_course_style()

## 1. Generators: one tick at a time

In [ ]:
import sys
ticks_list = list(p.tick_stream(50_000))
print(f"list of 50,000 ticks: {sys.getsizeof(ticks_list) / 1e6:.1f} MB for the list alone (plus every tuple)")
gen = p.tick_stream(50_000)
print(f"the generator:        {sys.getsizeof(gen)} bytes, whatever the length")
print(next(gen), next(gen), sep="\n")

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def bars_from_ticks(ticks, minutes=1):
    cur = None
    for t, px, sz in ticks:
        start = t.floor(f"{minutes}min")
        if cur is not None and start != cur["start"]:
            yield cur                          # the previous bar is complete
            cur = None
        if cur is None:
            cur = {"start": start, "open": px, "high": px, "low": px, "close": px, "volume": 0}
        # ✍️ update high, low, close and volume with this tick
        ...
    if cur is not None:
        yield cur

mine = pd.DataFrame(bars_from_ticks(p.tick_stream(3000), minutes=5))
bars = p.check("bars from a tick stream", mine, pd.DataFrame(p.bars_from_ticks(p.tick_stream(3000), minutes=5)))
bars.head()

In [ ]:
ax = bars.set_index("start")["close"].plot(title="5-minute closes built from a tick stream")
ax.set_xlabel(""); plt.show()

## 2. Decorators

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
import functools

def count_calls(fn):
    @functools.wraps(fn)                  # keeps fn's name and docstring on the wrapper
    def wrapper(*args, **kwargs):
        # ✍️ increase wrapper.calls by one, then call fn and return its result
        return ...
    wrapper.calls = 0
    return wrapper

@count_calls
def fetch_quote(symbol):
    """Pretend broker call."""
    return {"symbol": symbol, "bid": 99.99, "ask": 100.01}

for s in ("SPY", "QQQ", "IWM"):
    fetch_quote(s)
last = fetch_quote("TLT")
result = (fetch_quote.calls, fetch_quote.__name__, last)
result = p.check("count_calls decorator", result, (4, "fetch_quote", {"symbol": "TLT", "bid": 99.99, "ask": 100.01}))
result

## 3. Context managers

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
import time
from contextlib import contextmanager

@contextmanager
def timer(results: dict, name: str):
    t0 = time.perf_counter()
    try:
        yield
    finally:
        # ✍️ store the elapsed seconds in results[name] (runs even if the block raises)
        ...

timings = {}
with timer(timings, "sleep"):
    time.sleep(0.05)
ok = p.check("timer context manager", 0.04 < timings.get("sleep", 0) < 0.5, True)

In [ ]:
def retry(times=3, delay=0.01, exceptions=(ConnectionError,)):
    """Retry on the given exceptions with a doubling delay; re-raise after the last try."""
    def deco(fn):
        @functools.wraps(fn)
        def wrapper(*a, **k):
            wait = delay
            for attempt in range(1, times + 1):
                try:
                    return fn(*a, **k)
                except exceptions as e:
                    if attempt == times:
                        raise
                    print(f"  attempt {attempt} failed ({e}); retrying in {wait:.2f}s")
                    time.sleep(wait)
                    wait *= 2
        return wrapper
    return deco

attempts = iter([ConnectionError("gateway down"), ConnectionError("timeout"), "connected"])

@retry(times=3)
def connect():
    x = next(attempts)
    if isinstance(x, Exception):
        raise x
    return x

print(connect())

## Questions
1. Why is a generator a natural fit for a live data feed?
2. What would `fetch_quote.__name__` be without `functools.wraps`, and why does it matter in logs?
3. Which exceptions should a broker retry catch, and which must never be retried (hint: an order rejection)?

**Graded version:** `labs/part03/week08_advanced` (tick generator → bars, `count_calls`, async `retry`).